# Planificar de verdad: el mundo de bloques

**Facsímil 2 · Inteligencia clásica** — capítulos 9 y 10 (planificación automática, PDDL y modelado
de dominios).

Hay una diferencia enorme entre *ejecutar* una secuencia de pasos y *deducir* cuál debe ser esa
secuencia. Lo segundo es **planificar**, y es lo que hace una IA cuando tiene que actuar en el mundo:
mover archivos, reservar recursos, operar una máquina, montar un mueble siguiendo unas piezas, cargar
un camión sin que se caiga la pila, o —hoy— cuando un agente con un modelo de lenguaje «planea pasos»
con sus herramientas. En este cuaderno le das a un planificador un montón de bloques desordenados y una
foto de cómo los quieres, **sin decirle los pasos**. Solo describes las acciones posibles con sus
precondiciones y efectos, y él arma el plan. Esa separación entre «qué se puede hacer» y «cómo
encadenarlo» es la idea central de PDDL, el lenguaje estándar de la planificación automática.

### Qué vas a aprender
- Cómo se representa un problema de planificación: **estados** (conjuntos de hechos), **acciones**
  (precondiciones + efectos) y una **meta**.
- Que planificar es, otra vez, **buscar en un espacio de estados** (¡como el primer cuaderno!), pero
  ahora los estados son situaciones del mundo.
- Cómo un planificador **deduce** el orden correcto sin que se lo programes, y cómo **dibujarlo**.
- Cómo el espacio de estados **explota** al añadir objetos, y cómo una **heurística** (A* y búsqueda
  voraz) recorta esa explosión sin que dejes de obtener un plan.
- Por qué un caso pequeñísimo de tres bloques tiene nombre propio en la historia de la IA: la
  **anomalía de Sussman**.

### Cuánto cuesta
Unos 15 minutos. CPU, solo Python estándar y matplotlib para los dibujos.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
from collections import deque
import time

# Un estado es un conjunto de hechos verdaderos (predicados).
# Bloques A, B, C. Predicados: ('sobre',x,y) ('mesa',x) ('libre',x)
#                              ('mano_vacia',) ('cogido',x)
inicio = frozenset({('sobre','C','A'), ('mesa','A'), ('mesa','B'),
                    ('libre','C'), ('libre','B'), ('mano_vacia',)})
meta = {('sobre','A','B'), ('sobre','B','C')}   # torre A encima de B encima de C

def dibuja(estado):
    encima = {h[2]: h[1] for h in estado if h[0] == 'sobre'}
    bases = [h[1] for h in estado if h[0] == 'mesa']
    cols = []
    for base in sorted(bases):
        torre = [base]
        while torre[-1] in encima: torre.append(encima[torre[-1]])
        cols.append(torre)
    print("  ".join("/".join(reversed(c)) for c in cols), " (sobre la mesa)")
print("ESTADO INICIAL:"); dibuja(inicio)
print("META: torre A/B/C (A arriba del todo)")


## 1. El estado: una foto del mundo como conjunto de hechos

En planificación clásica (el enfoque **STRIPS**, base de PDDL), un estado no es una imagen ni una
estructura complicada: es simplemente el **conjunto de hechos que son verdaderos** en ese momento. Por
ejemplo, «C está sobre A», «A está sobre la mesa», «la mano está vacía». Todo lo que no está en el
conjunto se asume falso. Esta representación tan austera es lo que hace la planificación tratable: un
estado es un conjunto, y comparar o transformar conjuntos es barato.


## 2. Las acciones: precondiciones y efectos

Aquí está el corazón de PDDL. Cada acción se describe con tres cosas:

- **Precondiciones:** qué hechos deben ser verdaderos para poder ejecutarla.
- **Efectos de borrado:** qué hechos dejan de ser verdaderos al ejecutarla.
- **Efectos de adición:** qué hechos pasan a ser verdaderos.

Por ejemplo, *coger un bloque de la mesa* requiere que esté libre, sobre la mesa y la mano vacía;
borra esos hechos y añade «cogido». **No hay ninguna instrucción sobre el orden**: el planificador lo
descubrirá. Definimos las cuatro acciones del mundo de bloques. Ordenamos la lista de acciones por
nombre solo para que el resultado sea **reproducible** (sin eso, el orden dependería de detalles
internos de Python y los recuentos cambiarían en cada ejecución).


In [ ]:
def acciones(estado):
    hechos = estado
    libres = {h[1] for h in hechos if h[0] == 'libre'}
    mano_vacia = ('mano_vacia',) in hechos
    cogido = next((h[1] for h in hechos if h[0] == 'cogido'), None)
    bloques = {h[1] for h in hechos if h[0] in ('mesa','libre','cogido')}
    res = []
    if mano_vacia:
        for x in libres:
            if ('mesa', x) in hechos:                       # COGER de la mesa
                nuevo = set(hechos) - {('mesa',x),('libre',x),('mano_vacia',)} | {('cogido',x)}
                res.append((f"coger {x} de la mesa", frozenset(nuevo)))
            for y in bloques:                               # DESAPILAR x de y
                if ('sobre',x,y) in hechos:
                    nuevo = set(hechos) - {('sobre',x,y),('libre',x),('mano_vacia',)} | {('cogido',x),('libre',y)}
                    res.append((f"desapilar {x} de {y}", frozenset(nuevo)))
    if cogido:
        nuevo = set(hechos) - {('cogido',cogido)} | {('mesa',cogido),('libre',cogido),('mano_vacia',)}
        res.append((f"dejar {cogido} en la mesa", frozenset(nuevo)))   # DEJAR en la mesa
        for y in libres:                                    # APILAR cogido sobre y
            nuevo = set(hechos) - {('cogido',cogido),('libre',y)} | {('sobre',cogido,y),('libre',cogido),('mano_vacia',)}
            res.append((f"apilar {cogido} sobre {y}", frozenset(nuevo)))
    res.sort(key=lambda t: t[0])   # orden estable -> resultados reproducibles
    return res

print("Acciones aplicables ahora mismo desde el estado inicial:")
for nombre, _ in acciones(inicio): print("  -", nombre)


## 3. Planificar = buscar el plan más corto

Una vez tenemos estados y acciones, planificar es **buscar**: desde el estado inicial, aplica acciones
posibles, generando nuevos estados, hasta alcanzar uno que cumpla la meta. Es exactamente la búsqueda
en anchura (BFS) del primer cuaderno de este facsímil, pero en un espacio donde los «nodos» son
situaciones del mundo. BFS nos garantiza el plan **más corto** en número de acciones. Contamos cuántos
estados explora.


In [ ]:
def planificar(inicio, meta):
    if meta <= inicio: return [], 1
    frontera = deque([(inicio, [])]); vistos = {inicio}; explorados = 0
    while frontera:
        estado, plan = frontera.popleft(); explorados += 1
        for nombre, nuevo in acciones(estado):
            if nuevo in vistos: continue
            if meta <= nuevo:
                return plan + [nombre], explorados
            vistos.add(nuevo); frontera.append((nuevo, plan + [nombre]))
    return None, explorados

plan, explorados = planificar(inicio, meta)
print(f"Plan encontrado en {len(plan)} pasos (explorando {explorados} estados):\n")
for i, paso in enumerate(plan, 1):
    print(f"  {i}. {paso}")


**Fíjate en lo que NO ha pasado.** En ningún momento le dijimos «primero quita C de encima de A».
Solo describimos las cuatro acciones con sus reglas, y el planificador **dedujo** que, para poner A
sobre B, antes hay que liberar A, y para eso quitar C. Eso es planificar: razonar la secuencia a partir
del modelo, no ejecutarla de memoria. Comprobemos que el plan, aplicado paso a paso, lleva de verdad a
la meta.


In [ ]:
estado = inicio
print("Ejecución del plan, paso a paso:")
dibuja(estado)
for paso in plan:
    estado = dict(acciones(estado))[paso]
    print(f"  tras '{paso}':"); dibuja(estado)
estado_final = estado                       # lo guardamos para dibujarlo
print("\n¿Cumple la meta?:", meta <= estado_final)


## 4. Verlo de verdad: el estado inicial y el final

El dibujo en texto está bien para depurar, pero una imagen lo cuenta de un vistazo. Dibujamos cada
torre como una pila de rectángulos: a la izquierda el desorden de partida (C estorbando sobre A), a la
derecha la torre A/B/C que el planificador ha conseguido. Misma función servirá luego para otros
problemas.


In [ ]:
import matplotlib.pyplot as plt

def columnas(estado):
    encima = {h[2]: h[1] for h in estado if h[0] == 'sobre'}
    bases = sorted(h[1] for h in estado if h[0] == 'mesa')
    cols = []
    for base in bases:
        torre = [base]
        while torre[-1] in encima: torre.append(encima[torre[-1]])
        cols.append(torre)
    return cols

def dibuja_mpl(estados_titulos):
    n = len(estados_titulos)
    fig, axes = plt.subplots(1, n, figsize=(3.2*n, 3.4))
    if n == 1: axes = [axes]
    for ax, (estado, titulo) in zip(axes, estados_titulos):
        cols = columnas(estado)
        for cx, torre in enumerate(cols):
            for ny, bloque in enumerate(torre):
                ax.add_patch(plt.Rectangle((cx*1.3, ny), 1, 0.9,
                             facecolor="#dcdcdc", edgecolor="black", lw=1.5))
                ax.text(cx*1.3+0.5, ny+0.45, bloque, ha="center", va="center",
                        fontsize=14, weight="bold")
        ancho = max(1, len(cols))
        ax.set_xlim(-0.4, ancho*1.3); ax.set_ylim(0, 5)
        ax.axhline(0, color="black", lw=2)        # la mesa
        ax.set_title(titulo); ax.axis("off")
    plt.tight_layout(); plt.show()

dibuja_mpl([(inicio, "inicial: C estorba sobre A"), (estado_final, "meta: torre A/B/C")])


## 5. Ese problema tiene nombre: la anomalía de Sussman

Lo que acabas de resolver no es un ejemplo cualquiera. Empezar con **C sobre A**, con A y B en la mesa,
y querer **A sobre B sobre C** es un caso clásico de la planificación: la **anomalía de Sussman**
(Gerald Sussman, años setenta). Es famoso porque hizo tropezar a los primeros planificadores, los que
intentaban lograr los subobjetivos **de uno en uno**:

- Si primero consigues «A sobre B», para hacerlo tienes A apoyado en B; pero luego, para colocar «B
  sobre C», tienes que **deshacer** lo anterior. Los subobjetivos **interfieren**.
- Si lo intentas al revés, pasa lo mismo por el otro lado.

No existe ningún orden de los dos subobjetivos que funcione tratándolos por separado: hay que
**entrelazar** los pasos. Nuestro planificador no cae en la trampa por una razón sencilla: no persigue
subobjetivos sueltos, sino que **busca en el espacio completo de estados**, así que encuentra
directamente la secuencia entrelazada correcta. La moraleja del capítulo 9 es justa esta: descomponer
un objetivo en metas independientes es tentador, pero a menudo es una mentira útil que se rompe.


## 6. El problema del tamaño: cuando el espacio explota

La planificación por búsqueda ciega tiene un talón de Aquiles: el espacio de estados **crece de forma
explosiva** con el número de objetos. Cada bloque nuevo multiplica las situaciones posibles. Lo
medimos: planificamos la misma idea (apilar todos los bloques en una torre) con 3, 4 y 5 bloques, y
miramos cuántos estados hay que explorar y cuánto tarda. Nos quedamos en 5 bloques con BFS: un paso más
y la búsqueda ciega se vuelve incómoda (justo lo que queremos demostrar).


In [ ]:
def problema_torre(n):
    # n bloques en la mesa; meta: torre B0 sobre B1 sobre ... sobre B(n-1)
    nombres = [f"B{i}" for i in range(n)]
    ini = set()
    for b in nombres: ini |= {('mesa', b), ('libre', b)}
    ini.add(('mano_vacia',))
    objetivo = {('sobre', nombres[i], nombres[i+1]) for i in range(n-1)}
    return frozenset(ini), objetivo

print("bloques | pasos del plan | estados explorados |  tiempo")
for n in [3, 4, 5]:
    ini, obj = problema_torre(n)
    t0 = time.perf_counter()
    pl, ex = planificar(ini, obj)
    dt = time.perf_counter() - t0
    print(f"   {n}    |      {len(pl):>3}       |      {ex:>7,}       | {dt:.3f}s")
print("\nCada bloque dispara el numero de estados: por eso la planificacion real usa HEURISTICAS")
print("(estimar 'cuanto falta'), igual que A* hacia con la busqueda en el primer cuaderno.")


**Esa tabla es la razón de ser del capítulo 10.** Con búsqueda ciega, pasar de 3 a 5 bloques
multiplica los estados explorados de forma brutal. Ningún problema real (con decenas de objetos) se
resuelve así. La solución es la misma que en el primer cuaderno: **heurísticas** que estimen cuánto
falta para la meta y guíen la búsqueda. La idea de fondo no cambia: planificar es buscar, y buscar bien
es buscar con criterio.


## 7. ¿Por qué explota? El factor de ramificación

Antes de meter heurísticas, conviene ver de dónde sale la explosión. En cada estado hay un puñado de
acciones aplicables; cada una abre una rama nueva del árbol de búsqueda. Ese número medio de ramas por
estado es el **factor de ramificación**, y es el motor de la combinatoria: si de media hay $b$ acciones
por estado, a profundidad $d$ el árbol tiene del orden de $b^d$ caminos. Vamos a medirlo de verdad
recorriendo todos los estados alcanzables de la torre de 4 bloques.


In [ ]:
print("Acciones aplicables desde el estado inicial (el caso Sussman):", len(acciones(inicio)))

# Recorremos TODO el espacio alcanzable de la torre-4 y medimos la media de acciones por estado.
ini4, _ = problema_torre(4)
vistos = {ini4}; cola = deque([ini4]); total = 0; n_estados = 0
while cola:
    e = cola.popleft(); acs = acciones(e); total += len(acs); n_estados += 1
    for _, nv in acs:
        if nv not in vistos: vistos.add(nv); cola.append(nv)
print(f"Torre de 4 bloques: {n_estados} estados alcanzables, "
      f"factor de ramificacion medio = {total/n_estados:.2f} acciones por estado")
print("Parece poco, pero elevado a la profundidad del plan es lo que dispara la busqueda ciega.")


## 8. Planificar con heurística: estimar cuánto falta

La idea es la misma que A* en el primer cuaderno: en vez de explorar a ciegas, **prioriza** los estados
que *parecen* más cerca de la meta. Necesitamos una **heurística** $h$ que estime el coste que queda.
La más simple aquí: **cuántos hechos de la meta todavía no se cumplen**, es decir `len(meta - estado)`.

Implementamos un planificador con una cola de prioridad (`heapq`) que ordena por $f = g + h$, donde $g$
es el coste real acumulado (pasos dados). Con un parámetro lo convertimos en dos algoritmos:

- **A*** ($f = g + h$): tiene en cuenta lo andado y lo que falta. Si $h$ nunca se pasa de optimista
  (es *admisible*), A* devuelve el plan **óptimo**.
- **Búsqueda voraz** (*best-first*, $f = h$): ignora lo andado y se lanza hacia lo que parece más cerca.
  Rapidísima, pero **sin garantía** de que el plan sea el más corto.

¿Es admisible nuestra $h$? En esta formulación, cada hecho de la meta pendiente exige al menos una
acción de «apilar» para lograrse, así que $h$ nunca sobreestima: es admisible y A* conserva el óptimo.
Cuidado con generalizar: contar subobjetivos pendientes deja de ser admisible en dominios donde **una
sola acción logra varios a la vez** (los contaría por separado y se pasaría).


In [ ]:
import heapq

def planificar_heur(inicio, meta, usar_g=True):
    # usar_g=True  -> A* (f = g + h), optimo si h es admisible
    # usar_g=False -> busqueda voraz (f = h), rapida pero sin garantia de optimo
    def h(estado): return len(meta - estado)
    cont = 0                                  # desempate estable para el heap
    abierta = [(h(inicio), 0, cont, inicio, [])]
    mejor_g = {inicio: 0}; explorados = 0
    while abierta:
        f, g, _, estado, plan = heapq.heappop(abierta)
        if g > mejor_g.get(estado, 1e9): continue
        explorados += 1
        if meta <= estado:
            return plan, explorados
        for nombre, nuevo in acciones(estado):
            ng = g + 1
            if ng < mejor_g.get(nuevo, 1e9):
                mejor_g[nuevo] = ng; cont += 1
                prioridad = (ng + h(nuevo)) if usar_g else h(nuevo)
                heapq.heappush(abierta, (prioridad, ng, cont, nuevo, plan + [nombre]))
    return None, explorados

p_bfs, e_bfs = planificar(inicio, meta)
p_astar, e_astar = planificar_heur(inicio, meta, usar_g=True)
p_voraz, e_voraz = planificar_heur(inicio, meta, usar_g=False)
print("El mismo caso Sussman, con tres estrategias:")
print(f"  BFS ciego        : {len(p_bfs)} pasos, {e_bfs} estados explorados")
print(f"  A* (g + h)       : {len(p_astar)} pasos, {e_astar} estados explorados")
print(f"  Voraz (solo h)   : {len(p_voraz)} pasos, {e_voraz} estados explorados")
print("\nEn un problema tan pequeno los tres encuentran un plan de 6 pasos; el ahorro de A* y voraz")
print("apenas se nota. La diferencia se ve cuando el problema crece (lo medimos justo despues).")


## 9. La heurística, en acción: ¿qué «ve» el planificador?

Una heurística buena es la que baja de forma clara según te acercas a la meta. Miremos el valor de $h$
(hechos de la meta aún sin cumplir) a lo largo del plan que ya encontramos. Verás algo revelador.


In [ ]:
def h(estado): return len(meta - estado)
estado = inicio
print(f"  h al empezar = {h(inicio)}   (ninguno de los 2 hechos de la meta se cumple aun)")
for paso in p_bfs:
    estado = dict(acciones(estado))[paso]
    print(f"  tras '{paso}': h = {h(estado)}")
print("\nLa h se queda ESTANCADA en 2 casi todo el plan y solo baja al final: es una heuristica floja.")
print("Por eso la voraz puede despistarse y dar planes mas largos: durante el llano, no la guia nadie.")


## 10. La prueba en la torre: BFS frente a A* y voraz

Aquí se ve la utilidad de la heurística. Comparamos las tres estrategias en la torre de bloques. Para
los tamaños donde BFS aún es practicable (3, 4, 5) ponemos las tres; para 6 y 7 bloques, donde la
búsqueda ciega ya sería incómoda, dejamos solo A* y la voraz. Fíjate en dos cosas: cuántos estados
explora cada una y **cuántos pasos** tiene el plan que devuelve.


In [ ]:
print("              BFS ciego        A* (optimo)       voraz (rapida)")
print("   n  |    pasos  estados |   pasos  estados |   pasos  estados")
datos = []
for n in [3, 4, 5]:
    ini, obj = problema_torre(n)
    pb, eb = planificar(ini, obj)
    pa, ea = planificar_heur(ini, obj, usar_g=True)
    pv, ev = planificar_heur(ini, obj, usar_g=False)
    datos.append((n, eb, ea, ev))
    print(f"   {n}  |     {len(pb):>3}   {eb:>5} |     {len(pa):>3}   {ea:>5} |     {len(pv):>3}   {ev:>5}")

print("\n   Tamanos donde la BFS ciega ya no compensa (solo A* y voraz):")
print("   n  |   A*: pasos  estados |  voraz: pasos  estados")
for n in [6, 7]:
    ini, obj = problema_torre(n)
    pa, ea = planificar_heur(ini, obj, usar_g=True)
    pv, ev = planificar_heur(ini, obj, usar_g=False)
    print(f"   {n}  |        {len(pa):>3}   {ea:>6} |          {len(pv):>3}   {ev:>5}")

# Grafico: estados explorados por estrategia (escala logaritmica)
ns = [d[0] for d in datos]
x = range(len(ns)); w = 0.27
fig, ax = plt.subplots(figsize=(5.8, 3.5))
ax.bar([i-w for i in x], [d[1] for d in datos], w, label="BFS ciego", color="#444444")
ax.bar([i   for i in x], [d[2] for d in datos], w, label="A* (optimo)", color="#999999")
ax.bar([i+w for i in x], [d[3] for d in datos], w, label="voraz", color="#dddddd", edgecolor="black")
ax.set_yscale("log"); ax.set_xticks(list(x)); ax.set_xticklabels([f"{n} bloques" for n in ns])
ax.set_ylabel("estados explorados (escala log)")
ax.set_title("Coste de la busqueda segun la estrategia"); ax.legend()
plt.tight_layout(); plt.show()


**Lee la tabla con calma, porque cuenta toda la historia del capítulo 10:**

- **A* explora menos que BFS y devuelve el mismo plan óptimo.** Con 5 bloques, BFS revienta cientos de
  estados y A* se queda muy por debajo, y los dos dan un plan de 8 pasos. Eso es la heurística trabajando:
  el óptimo sale más barato.
- **La voraz explora poquísimo, pero lo paga en la longitud del plan.** Apenas mira un puñado de estados,
  pero sus planes son notablemente más largos que el óptimo (a veces casi el doble). Va deprisa, no va
  bien.
- **A escala, A* es lo que sigue en pie.** Con 6 y 7 bloques la búsqueda ciega ya sería incómoda;
  A* sigue dando el plan más corto con un coste razonable.

La lección práctica: una heurística admisible te compra **velocidad sin perder optimalidad**; relajar la
admisibilidad (la voraz, o A* ponderado) te compra **aún más velocidad a cambio de calidad**. Elegir en
esa balanza es buena parte del oficio de la planificación.


## 11. El mismo motor, otro problema: intercambiar dos bloques

Nada de lo que hicimos es específico de «hacer torres». El planificador solo conoce acciones, estados y
metas, así que le podemos dar **otro problema** sin tocar el motor. Probemos un intercambio: partimos de
**A sobre B** (con C aparte en la mesa) y queremos justo lo contrario, **B sobre A**. Lo interesante es
que para darle la vuelta a la pila hace falta un **sitio temporal** donde aparcar un bloque: igual que
para sacar el plato de abajo de una pila hay que mover los de encima a otro lado.


In [ ]:
swap_ini = frozenset({('sobre','A','B'), ('mesa','B'), ('mesa','C'),
                      ('libre','A'), ('libre','C'), ('mano_vacia',)})
swap_meta = {('sobre','B','A')}

plan_swap, ex_swap = planificar(swap_ini, swap_meta)
print(f"Intercambio A/B -> B/A: plan de {len(plan_swap)} pasos ({ex_swap} estados explorados):")
for i, paso in enumerate(plan_swap, 1): print(f"  {i}. {paso}")

# Validamos que el plan, ejecutado, alcanza de verdad la meta
estado = swap_ini
for paso in plan_swap: estado = dict(acciones(estado))[paso]
print("\n¿El plan cumple la meta?:", swap_meta <= estado)
dibuja_mpl([(swap_ini, "inicial: A sobre B"), (estado, "meta: B sobre A")])


Mira el segundo paso del plan: el planificador **usa el bloque C (o la mesa) como hueco temporal**
para poder dar la vuelta a la pila. Nadie se lo dijo; lo dedujo de las precondiciones. Esa capacidad de
inventar pasos «de apoyo» que no aparecen en la meta es justo lo que distingue planificar de seguir una
receta.


## 12. Modelar el dominio es el verdadero trabajo

El algoritmo de búsqueda es el mismo siempre; lo que cambia de un problema a otro es el **modelo**: las
precondiciones y los efectos. Y ahí está el 90% de los errores reales. Un efecto de borrado olvidado
genera estados imposibles, y el planificador, tan obediente, los dará por buenos. Lo vemos con una
versión **rota** de «coger de la mesa» a la que se le olvida borrar dos hechos.


In [ ]:
def coger_roto(estado, x):
    # VERSION CON BUG: anade 'cogido' pero se OLVIDA de borrar ('mesa',x) y ('mano_vacia',)
    return frozenset(set(estado) | {('cogido', x)})

malo = coger_roto(inicio, 'B')
print("Tras el 'coger' roto, el estado dice a la vez que B esta...")
print("  - cogido en la mano:", ('cogido','B') in malo)
print("  - y todavia sobre la mesa:", ('mesa','B') in malo)
print("  - y la mano sigue 'vacia':", ('mano_vacia',) in malo)
print("\nEstado fisicamente imposible. El planificador no lo detecta: confia en tu modelo.")
print("Por eso modelar bien (precondiciones y efectos completos) es donde se gana o se pierde.")


## 13. Dónde encaja esto hoy: agentes que planifican

Esto no es una reliquia. La misma estructura —estados, acciones con precondiciones y efectos, y una
búsqueda con heurística— aparece cada vez que algo tiene que **decidir una secuencia de pasos**:

- **Montar un mueble** o **cargar un camión**: orden de piezas con restricciones (no puedes atornillar
  una pata si antes no has dado la vuelta al tablero; no apilas lo frágil debajo).
- **Logística y robótica**: planificar rutas, manipular objetos, secuenciar una cadena de montaje.
- **Agentes con modelos de lenguaje**: un agente que «razona pasos» con herramientas está, en el fondo,
  proponiendo un plan. Lo potente es combinar ambos mundos: el LLM **sugiere** un plan plausible y un
  verificador como el de este cuaderno comprueba que cada paso cumple sus precondiciones y, si el mundo
  no responde como se esperaba, **replanifica**. El facsímil 5 lleva esta idea a agentes reales.


## 14. Pruébalo tú

1. **Cambia la meta** a `{('sobre','C','B'),('sobre','B','A')}` (la torre al revés). ¿Cuántos pasos y
   cuántos estados con BFS? ¿Y con A* (`planificar_heur(inicio, nueva_meta)`)? ¿Cuánto ahorra la heurística?
2. **Añade un bloque D** al estado inicial y mételo en la torre. Compara `planificar` (BFS) con
   `planificar_heur` en estados explorados: estás tocando con el dedo por qué hacen falta heurísticas.
3. **Vuelve voraz a A*** (`usar_g=False`) en la torre de 5 o 6 bloques y compara la **longitud** del plan
   con la de A*. ¿Cuánto se aleja del óptimo a cambio de ir más rápida?
4. **Inventa tu propio problema**: tres bloques en una configuración cualquiera y una meta distinta.
   Dibújalo con `dibuja_mpl` antes y después. El motor no cambia, solo el estado inicial y la meta.
5. **Rompe el modelo a propósito**: quita una precondición de `apilar` (por ejemplo, deja apilar sobre un
   bloque que no esté libre) y mira cómo el planificador encuentra «planes» imposibles.


## 15. Errores comunes

- **Olvidar un efecto de borrado.** Lo acabamos de ver: si «coger» no borra «sobre la mesa», el estado
  queda inconsistente. Los efectos de borrado son tan importantes como los de adición.
- **Confundir el plan con su ejecución.** El planificador *deduce* la secuencia; ejecutarla es otra fase.
  Separar ambas cosas es justo lo que da robustez a un agente.
- **Esperar que la búsqueda ciega escale.** No lo hace. En cuanto hay varios objetos, hacen falta
  heurísticas (A*, voraz) o técnicas como traducir a SAT.
- **Confiar en la voraz porque «va rapidísima».** Va rápida, pero sus planes pueden ser muy largos. Si la
  optimalidad importa, A* con una heurística admisible es la apuesta segura.
- **Descomponer la meta en subobjetivos independientes.** La anomalía de Sussman enseña que los
  subobjetivos interfieren: a menudo hay que entrelazar los pasos, no resolverlos por separado.
- **Modelar mal el dominio.** El 90% de los fallos de un planificador son precondiciones o efectos mal
  escritos, no el algoritmo. Modelar bien es el verdadero trabajo.


## 16. Qué te llevas

- **Planificar = modelar acciones** (precondiciones + efectos) y **buscar** la secuencia que lleva del
  estado inicial a la meta. El modelo es declarativo; el plan, emergente.
- Es **la misma búsqueda** del primer cuaderno, ahora sobre estados del mundo. Y la **misma heurística**:
  A* y la búsqueda voraz recortan la explosión combinatoria, con un compromiso claro entre velocidad y
  optimalidad.
- Un problema de tres bloques (la **anomalía de Sussman**) basta para ver que descomponer metas en partes
  independientes es una trampa: los subobjetivos interfieren.
- El motor es **general**: torres, intercambios o lo que modeles, sin tocar el algoritmo. Lo que cambia
  —y donde se gana o se pierde— es el **modelo del dominio**.

**Para seguir:** el capítulo 10 profundiza en heurísticas y en traducir la planificación a SAT; el
facsímil 5 lleva esta idea a agentes que planifican y actúan con herramientas reales, donde el «plan» se
ejecuta paso a paso y se replanifica si el mundo no responde como se esperaba.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*